In [1]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
# # Solicitará permissão para acessar o seu Google Drive se for executado no Colab
# from google.colab import drive
# drive.mount('/content/drive')

# # Definindo o caminho da nova pasta no Google Drive
# caminho_pasta_base = '/content/drive/MyDrive/Fatec/TCC/dataset'
# caminho_pasta_tratado = '/content/drive/MyDrive/Fatec/TCC/dataset tratado'

In [3]:
# Definindo os caminho da nova pasta no PC pessoal
caminho_pasta_base = '../dataset/lycos-cicids2017'
caminho_pasta_tratado = '../dataset tratado/lycos-cicids2017/Sem Redução de Dimensionalidade'

In [4]:
# Arquivos para o cenário de teste com ataque não visto no treino
nome_dados_treinamento = 'lycos_cicids2017_treinamento.csv'
nome_dados_teste = 'lycos_cicids2017_teste.csv'

nome_dados_treinamento_sem_portscan = 'lycos_cicids2017_treinamento_sem_portscan.csv'
nome_dados_teste_com_portscan = 'lycos_cicids2017_teste_com_portscan.csv'

nome_dados_treinamento_sem_XSS = 'lycos_cicids2017_treinamento_sem_XSS.csv'
nome_dados_teste_com_XSS = 'lycos_cicids2017_teste_com_XSS.csv'

In [5]:
# Mapeia todos os arquivos que terminam com .csv dentro da pasta
arquivos_csv = glob.glob(os.path.join(caminho_pasta_base, "*.csv"))

print(f"Encontrados {len(arquivos_csv)} arquivos para importação.")

def ler_csv_com_encoding(arquivo):
    try:
        return pd.read_csv(arquivo, encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(arquivo, encoding="cp1252", low_memory=False)

# Lê cada um dos 8 arquivos e os junta em um único DataFrame
lista_dfs = []
for arquivo in arquivos_csv:
    df_temp = ler_csv_com_encoding(arquivo)
    lista_dfs.append(df_temp)

# Concatena todos os DataFrames da lista empilhando as linhas
df = pd.concat(lista_dfs, ignore_index=True)

# Remove espaços em branco no início/fim dos nomes das colunas
df.columns = df.columns.str.strip()
df_bruto = df.copy()

print("Tamanho total do dataset unificado:", df.shape)
display(df.head())

Encontrados 5 arquivos para importação.
Tamanho total do dataset unificado: (1837498, 83)


,flow_id,src_addr,src_port,dst_addr,dst_port,ip_prot,timestamp,flow_duration,down_up_ratio,pkt_len_max,...,bwd_bulk_bytes_mean,bwd_bulk_pkt_mean,bwd_bulk_rate_mean,fwd_subflow_bytes_mean,fwd_subflow_pkt_mean,bwd_subflow_bytes_mean,bwd_subflow_pkt_mean,fwd_tcp_init_win_bytes,bwd_tcp_init_win_bytes,label
0,192.168.10.3-192.168.10.50-3268-56108-6,192.168.10.3,3268,192.168.10.50,56108,6,1499428790315195,112740690,2.0,403.0,...,0.0,0.0,0.0,144.000000,2.000000,806.000000,4.000000,2078,377,benign
1,192.168.10.50-192.168.10.3-42144-389-6,192.168.10.50,42144,192.168.10.3,389,6,1499428790316273,112740560,0.5,403.0,...,0.0,0.0,0.0,806.000000,4.000000,632.000000,2.000000,955,2078,benign
2,192.168.10.9-224.0.0.22-0-0-2,192.168.10.9,0,224.0.0.22,0,2,1499428834843793,54760,0.0,8.0,...,0.0,0.0,0.0,32.000000,4.000000,0.000000,0.000000,-1,-1,benign
3,192.168.10.9-224.0.0.252-63210-5355-17,192.168.10.9,63210,224.0.0.252,5355,17,1499428834845380,100126,0.0,28.0,...,0.0,0.0,0.0,616.000000,22.000000,0.000000,0.000000,-1,-1,benign
4,192.168.10.9-192.168.10.3-137-137-17,192.168.10.9,137,192.168.10.3,137,17,1499428835164943,93069427,1.0,68.0,...,0.0,0.0,0.0,131.111111,2.222222,131.111111,2.222222,-1,-1,benign


In [6]:
# Converte valores infinitos para NaN (nulos)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Remove todas as linhas que contenham valores nulos
df.dropna(inplace=True)

print("Tamanho do dataset após remoção de nulos/infinitos:", df.shape)

Tamanho do dataset após remoção de nulos/infinitos: (1837498, 83)


In [7]:
from sklearn.preprocessing import MinMaxScaler

# Usa sempre a base original importada, evitando erro ao reexecutar esta célula
df = df_bruto.copy()
df.columns = df.columns.str.strip()

# Separando a variável alvo (Label) dos dados de tráfego
coluna_label = next((col for col in df.columns if col.strip().lower() == 'label'), None)
if coluna_label is None:
    raise KeyError(f"Coluna Label não encontrada. Colunas disponíveis: {list(df.columns)}")

label = df[coluna_label]

# Remove colunas textuais/identificadoras que não devem ser normalizadas
colunas_textuais = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']
df = df.drop(columns=[coluna_label] + colunas_textuais, errors='ignore')

# Garante que as features restantes sejam numéricas
df = df.apply(pd.to_numeric, errors='coerce')
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Remove colunas que são inteiramente NaN (colunas não-numéricas não removidas acima)
colunas_todas_nan = df.columns[df.isna().all()].tolist()
if colunas_todas_nan:
    print(f"Colunas não-numéricas removidas: {colunas_todas_nan}")
    df = df.drop(columns=colunas_todas_nan)

linhas_validas = df.notna().all(axis=1)
df = df.loc[linhas_validas].copy()
label = label.loc[linhas_validas].copy()

# Aplicação da escala Min-Max conforme exigido para convergência da Rede Neural
scaler = MinMaxScaler()
df = pd.DataFrame(
    scaler.fit_transform(df), 
    columns=df.columns,
    index=df.index
) 

# Readicionando a coluna 'Label' ao DataFrame tratado
df['Label'] = label.values

print("Dados numéricos normalizados com sucesso.")
display(df.head())

Colunas não-numéricas removidas: ['flow_id', 'src_addr', 'dst_addr']
Dados numéricos normalizados com sucesso.


,src_port,dst_port,ip_prot,timestamp,flow_duration,down_up_ratio,pkt_len_max,pkt_len_min,pkt_len_mean,pkt_len_var,...,bwd_bulk_bytes_mean,bwd_bulk_pkt_mean,bwd_bulk_rate_mean,fwd_subflow_bytes_mean,fwd_subflow_pkt_mean,bwd_subflow_bytes_mean,bwd_subflow_pkt_mean,fwd_tcp_init_win_bytes,bwd_tcp_init_win_bytes,Label
0,0.049866,0.856232,0.038168,0.922700,0.939506,0.247818,0.016237,0.000000,0.066210,0.001171,...,0.0,0.0,0.0,0.000081,0.000009,1.229685e-06,0.000014,0.031723,0.005768,benign
1,0.643076,0.005936,0.038168,0.922700,0.939505,0.061955,0.016237,0.000000,0.100221,0.001140,...,0.0,0.0,0.0,0.000452,0.000018,9.642196e-07,0.000007,0.014587,0.031723,benign
2,0.000000,0.000000,0.007634,0.922818,0.000456,0.000000,0.000322,0.005525,0.003345,0.000000,...,0.0,0.0,0.0,0.000018,0.000018,0.000000e+00,0.000000,0.000000,0.000000,benign
3,0.964523,0.081720,0.122137,0.922818,0.000834,0.000000,0.001128,0.019337,0.011709,0.000000,...,0.0,0.0,0.0,0.000345,0.000100,0.000000e+00,0.000000,0.000000,0.000000,benign
4,0.002090,0.002091,0.122137,0.922819,0.775579,0.123909,0.002740,0.034530,0.024672,0.000002,...,0.0,0.0,0.0,0.000073,0.000010,2.000315e-07,0.000008,0.000000,0.000000,benign


In [8]:
from sklearn.model_selection import train_test_split

df_treino, df_teste = train_test_split(df, test_size=0.3, random_state=42)

print(f"Dados de Treino: {df_treino.shape}")
print(f"Dados de Teste: {df_teste.shape}")

# O comando makedirs cria a pasta "dataset filtrado" caso ela ainda não exista no seu Drive
os.makedirs(caminho_pasta_tratado, exist_ok=True)

# 3. Caminho completo do arquivo CSV que será gerado
caminho_arquivo_treinamento = os.path.join(caminho_pasta_tratado, nome_dados_treinamento)
caminho_arquivo_teste = os.path.join(caminho_pasta_tratado, nome_dados_teste)

# 4. Exportando o DataFrame para o formato CSV
df_treino.to_csv(caminho_arquivo_treinamento, index=False)
print(f"Dataset de treinamento salvo com sucesso em: '{caminho_arquivo_treinamento}'")

df_teste.to_csv(caminho_arquivo_teste, index=False)
print(f"Dataset de teste salvo com sucesso em: '{caminho_arquivo_teste}'")

Dados de Treino: (1286248, 80)
Dados de Teste: (551250, 80)
Dataset de treinamento salvo com sucesso em: '../dataset tratado/lycos-cicids2017/Sem Redução de Dimensionalidade\lycos_cicids2017_treinamento.csv'
Dataset de teste salvo com sucesso em: '../dataset tratado/lycos-cicids2017/Sem Redução de Dimensionalidade\lycos_cicids2017_teste.csv'


In [9]:
# Colocando PortScan somente no teste para simular o cenário de ataque não visto no treino
df_treino_sem_portscan, df_teste_com_portscan = train_test_split(df, test_size=0.3, random_state=42)

# Move todos os registros PortScan do treino para o teste
eh_portscan_treino = df_treino_sem_portscan['Label'].astype(str).str.contains('portscan', case=False, na=False)
portscan_do_treino = df_treino_sem_portscan[eh_portscan_treino].copy()
df_treino_sem_portscan = df_treino_sem_portscan[~eh_portscan_treino].copy()
df_teste_com_portscan = pd.concat([df_teste_com_portscan, portscan_do_treino], ignore_index=True)

print(f"PortScan movido do treino para o teste: {len(portscan_do_treino)}")
print(f"PortScan no treino após mover: {df_treino_sem_portscan['Label'].astype(str).str.contains('portscan', case=False, na=False).sum()}")
print(f"PortScan no teste após mover: {df_teste_com_portscan['Label'].astype(str).str.contains('portscan', case=False, na=False).sum()}")

print(f"Dados de Treino: {df_treino_sem_portscan.shape}")
print(f"Dados de Teste: {df_teste_com_portscan.shape}")

os.makedirs(caminho_pasta_tratado, exist_ok=True)

caminho_arquivo_treinamento = os.path.join(caminho_pasta_tratado, nome_dados_treinamento_sem_portscan)
caminho_arquivo_teste = os.path.join(caminho_pasta_tratado, nome_dados_teste_com_portscan)

df_treino_sem_portscan.to_csv(caminho_arquivo_treinamento, index=False)
print(f"Dataset de treinamento salvo com sucesso em: {caminho_arquivo_treinamento}")

df_teste_com_portscan.to_csv(caminho_arquivo_teste, index=False)
print(f"Dataset de teste salvo com sucesso em: {caminho_arquivo_teste}")

PortScan movido do treino para o teste: 112199
PortScan no treino após mover: 0
PortScan no teste após mover: 160106
Dados de Treino: (1174049, 80)
Dados de Teste: (663449, 80)
Dataset de treinamento salvo com sucesso em: ../dataset tratado/lycos-cicids2017/Sem Redução de Dimensionalidade\lycos_cicids2017_treinamento_sem_portscan.csv
Dataset de teste salvo com sucesso em: ../dataset tratado/lycos-cicids2017/Sem Redução de Dimensionalidade\lycos_cicids2017_teste_com_portscan.csv


In [10]:
# Colocando XSS somente no teste para simular o cenário de ataque não visto no treino
df_treino_sem_xss, df_teste_com_xss = train_test_split(df, test_size=0.3, random_state=42)

# Move todos os registros XSS do treino para o teste
eh_xss_treino = df_treino_sem_xss['Label'].astype(str).str.contains('XSS', case=False, na=False)
xss_do_treino = df_treino_sem_xss[eh_xss_treino].copy()
df_treino_sem_xss = df_treino_sem_xss[~eh_xss_treino].copy()
df_teste_com_xss = pd.concat([df_teste_com_xss, xss_do_treino], ignore_index=True)

print(f"XSS movido do treino para o teste: {len(xss_do_treino)}")
print(f"XSS no treino após mover: {df_treino_sem_xss['Label'].astype(str).str.contains('XSS', case=False, na=False).sum()}")
print(f"XSS no teste após mover: {df_teste_com_xss['Label'].astype(str).str.contains('XSS', case=False, na=False).sum()}")

print(f"Dados de Treino: {df_treino_sem_xss.shape}")
print(f"Dados de Teste: {df_teste_com_xss.shape}")

os.makedirs(caminho_pasta_tratado, exist_ok=True)

caminho_arquivo_treinamento = os.path.join(caminho_pasta_tratado, nome_dados_treinamento_sem_XSS)
caminho_arquivo_teste = os.path.join(caminho_pasta_tratado, nome_dados_teste_com_XSS)

df_treino_sem_xss.to_csv(caminho_arquivo_treinamento, index=False)
print(f"Dataset de treinamento salvo com sucesso em: {caminho_arquivo_treinamento}")

df_teste_com_xss.to_csv(caminho_arquivo_teste, index=False)
print(f"Dataset de teste salvo com sucesso em: {caminho_arquivo_teste}")

XSS movido do treino para o teste: 468
XSS no treino após mover: 0
XSS no teste após mover: 661
Dados de Treino: (1285780, 80)
Dados de Teste: (551718, 80)
Dataset de treinamento salvo com sucesso em: ../dataset tratado/lycos-cicids2017/Sem Redução de Dimensionalidade\lycos_cicids2017_treinamento_sem_XSS.csv
Dataset de teste salvo com sucesso em: ../dataset tratado/lycos-cicids2017/Sem Redução de Dimensionalidade\lycos_cicids2017_teste_com_XSS.csv
